# Lab Assignment 1 — Pandas vs. Polars
**ARTI 303 — Programming for AI**

| | |
|---|---|

---

## What this assignment is about

Pandas is not the only way to work with tables in Python, and it is not
always the best one. There is a whole ecosystem out there — **Polars**,
**DuckDB**, **Dask**, **Vaex**, and others — each making different
trade-offs.

The point of this assignment is not to prove one library beats another.
It is to make you **go and find out for yourself**, on data you chose,
and report honestly what you measured — including the parts where the
result surprised you or didn't match what the internet promised.

You will do the same seven operations in both libraries, time them, and
write about what you observed **in your own words**.

---

## What you will submit

**1. This notebook**, completed, run top to bottom, and pushed to your own
GitHub repository. Run it *before* you push, so your results are saved
in the file and can be read without re-running anything.

**2. A Word document** containing, in this order:

1. Your name, student ID, and section
2. A screenshot of you **pulling** this assignment
3. Your dataset link
4. Your GitHub repository link
5. A screenshot of you **pushing** your completed work

### The two screenshots

Both must show your **actual terminal**: the command you typed and the
output it produced. A screenshot of the GitHub website is not accepted.


> Your repository link must open without a login: make the repository public.

---
# Part 1 · Setup

Install Polars into your course environment, with `(.venv)` active in your terminal, then run the cell below.

In [2]:
import pandas as pd
import polars as pl
import numpy as np
import time
import statistics

print("pandas:", pd.__version__)
print("polars:", pl.__version__)

pandas: 3.0.6
polars: 2.0.0


---
# Part 2 · Load your dataset

Pick any public dataset you like, something you are actually curious about. Requirements:

- **At least 100,000 rows.** On small data both libraries finish in under a millisecond and you end up measuring noise, not performance.
- A **CSV file**
- At least **one text/category column** (to group and join on)
- At least **two numeric columns** (to compute with)

Kaggle, data.gov.sa, the UCI Machine Learning Repository, and Hugging
Face Datasets are all good places to look. If the file is over about
50 MB, do not commit it: add it to `.gitignore` and put the download
link in your Word document.

Set `DATA_PATH` to your file, then run the cell.

In [3]:
DATA_PATH = "Records.csv"

df_pandas = pd.read_csv(DATA_PATH)
df_polars = pl.read_csv(DATA_PATH)

print("pandas shape:", df_pandas.shape)
print("polars shape:", df_polars.shape)
print()
print("columns:", list(df_pandas.columns))
print()
print(df_pandas.head(3))

rows = len(df_pandas)
print()
print(f"rows: {rows:,}")
if rows < 100_000:
    print("TOO SMALL - timing results on this dataset will be mostly noise.")
    print("Pick a larger dataset before continuing.")
else:
    print("Size is fine.")

pandas shape: (100000, 14)
polars shape: (100000, 14)

columns: ['Region', 'Country', 'Item Type', 'Sales Channel', 'Order Priority', 'Order Date', 'Order ID', 'Ship Date', 'Units Sold', 'Unit Price', 'Unit Cost', 'Total Revenue', 'Total Cost', 'Total Profit']

                              Region                Country  Item Type  \
0       Middle East and North Africa             Azerbaijan     Snacks   
1  Central America and the Caribbean                 Panama  Cosmetics   
2                 Sub-Saharan Africa  Sao Tome and Principe     Fruits   

  Sales Channel Order Priority Order Date   Order ID   Ship Date  Units Sold  \
0        Online              C  10/8/2014  535113847  10/23/2014         934   
1       Offline              L  2/22/2015  874708545   2/27/2015        4551   
2       Offline              M  12/9/2015  854349935   1/18/2016        9986   

   Unit Price  Unit Cost  Total Revenue  Total Cost  Total Profit  
0      152.58      97.44      142509.72    91008.96 

---
# Part 3 · How to time things properly


### Rule 1 — Run it more than once, take the median

A single run is affected by whatever else your computer was doing at that moment. Run each operation several times and take the middle value.

### Rule 2 — Polars is lazy. You must call `.collect()`

Polars has two modes. In **lazy** mode (`pl.scan_csv`, `.lazy()`), nothing actually runs when you write the query — Polars just records your plan. The work only happens when you call `.collect()`.

So if you time a lazy query without `.collect()`, you are timing the act of *writing down a plan*, not executing it. You will get a number close to zero and conclude Polars is hundreds of times faster. It is not. You measured nothing.

### Rule 3 — The first read of a file is slower

Your operating system caches file contents after the first read. Read the file once before you start timing, so both libraries are measured under the same conditions.

### The timing helper

Use this function for every measurement below.

In [4]:
def measure(operation, repeats=5):
    """Run `operation` several times and return the median time in milliseconds.

    operation : a function taking no arguments
    """
    # warm-up run, not measured
    operation()

    timings = []
    for _ in range(repeats):
        start = time.perf_counter()
        operation()
        timings.append(time.perf_counter() - start)

    return statistics.median(timings) * 1000

### See Rule 2 for yourself

Run this and look at the difference.

In [5]:
lazy_no_collect = measure(lambda: pl.scan_csv(DATA_PATH).filter(pl.col(df_pandas.columns[0]).is_not_null()))
lazy_collected  = measure(lambda: pl.scan_csv(DATA_PATH).filter(pl.col(df_pandas.columns[0]).is_not_null()).collect())

print(f"lazy WITHOUT .collect(): {lazy_no_collect:8.3f} ms   <- nothing was executed")
print(f"lazy WITH .collect():    {lazy_collected:8.3f} ms   <- the real cost")

lazy WITHOUT .collect():    0.022 ms   <- nothing was executed
lazy WITH .collect():      12.871 ms   <- the real cost


**Record what you saw:**

| | ms |
|---|---|
| Lazy query without `.collect()` | _[0.022 ms]_ |
| Lazy query with `.collect()` | _[12.871 ms]_ |

In one sentence — why is the first number meaningless?

_[Beacuse nothing was executed]_

---
# Part 4 · The seven operations

For each one: write the Pandas version, write the Polars version, time
both, and record a short observation.

Define which columns you are using once, here.

In [5]:
# Set these to real column names from YOUR dataset
CATEGORY_COL = "Item Type"   # a text/category column, for grouping
NUMERIC_COL = "Units Sold"    # a numeric column
NUMERIC_COL2 = "Total Profit" # a second numeric column
 

for col in [CATEGORY_COL, NUMERIC_COL, NUMERIC_COL2]:
    if col not in df_pandas.columns:
        print(f"[!] '{col}' is not in your dataset - update it above")
    else:
        print(f"[ok] {col}")

[ok] Item Type
[ok] Units Sold
[ok] Total Profit


## 4.1 — Reading the file

Time reading the whole file from disk in each library.

In [7]:
pandas_time = measure(lambda: pd.read_csv(DATA_PATH))
polars_time = measure(lambda: pl.read_csv(DATA_PATH))

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

read_pandas, read_polars = pandas_time, polars_time

pandas:    90.90 ms
polars:     6.95 ms
ratio:     13.08x


**Observation:** _[Polars was faster than Pandas when reading. Polars took 6.95 ms while Pandas took 90.90 ms, 13 times faster.]_

## 4.2 — Filtering rows

Keep only rows where your numeric column is above its median.

In [8]:
threshold = df_pandas[NUMERIC_COL].median()
print("threshold:", threshold)

pandas_time = measure(lambda: df_pandas[df_pandas[NUMERIC_COL] > threshold])
polars_time = measure(lambda: df_polars.filter(pl.col(NUMERIC_COL) > threshold))

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

filter_pandas, filter_polars = pandas_time, polars_time

threshold: 5007.0
pandas:     2.75 ms
polars:     1.77 ms
ratio:      1.55x


**Observation:** _[Polars was faster than Pandas for filtering rows above the median. Polars took 1.77 ms and Pandas took 2.75 ms, making it  1.55 times faster.]_

## 4.3 — Creating a new column

Compute a new column from two existing ones.

In [9]:
pandas_time = measure(
    lambda: df_pandas.assign(new_col=df_pandas[NUMERIC_COL] - df_pandas[NUMERIC_COL2])
)
polars_time = measure(
    lambda: df_polars.with_columns(
        (pl.col(NUMERIC_COL) - pl.col(NUMERIC_COL2)).alias("new_col")
    )
)

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

newcol_pandas, newcol_polars = pandas_time, polars_time

pandas:     4.56 ms
polars:     1.24 ms
ratio:      3.67x


**Observation:** _[Polars was faster than Pandas. Polars took 1.24 ms and to Pandas took 4.56 ms, making it 3.67 times faster.]_

## 4.4 — Group by and aggregate

Group by your category column and compute a sum and a mean.

In [10]:
pandas_time = measure(
    lambda: df_pandas.groupby(CATEGORY_COL).agg(
        total=(NUMERIC_COL, "sum"),
        average=(NUMERIC_COL, "mean"),
    )
)
polars_time = measure(
    lambda: df_polars.group_by(CATEGORY_COL).agg([
        pl.col(NUMERIC_COL).sum().alias("total"),
        pl.col(NUMERIC_COL).mean().alias("average"),
    ])
)

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

groupby_pandas, groupby_polars = pandas_time, polars_time

# Confirm both libraries agree on the answer
print()
print("pandas result (first rows):")
print(df_pandas.groupby(CATEGORY_COL)[NUMERIC_COL].sum().head())
print()
print("polars result (first rows):")
print(df_polars.group_by(CATEGORY_COL).agg(pl.col(NUMERIC_COL).sum()).head())

pandas:     5.95 ms
polars:     2.61 ms
ratio:      2.28x

pandas result (first rows):
Item Type
Baby Food    41911620
Beverages    41514213
Cereal       42254418
Clothes      41773440
Cosmetics    41924464
Name: Units Sold, dtype: int64

polars result (first rows):
shape: (5, 2)
┌───────────┬────────────┐
│ Item Type ┆ Units Sold │
│ ---       ┆ ---        │
│ str       ┆ i64        │
╞═══════════╪════════════╡
│ Meat      ┆ 41745367   │
│ Snacks    ┆ 41699092   │
│ Beverages ┆ 41514213   │
│ Household ┆ 41458795   │
│ Cosmetics ┆ 41924464   │
└───────────┴────────────┘


**Observation:** _[Polars was faster than Pandas for grouping and calculating. Polars took 2.61 ms and Pandas 5.95 ms, making it 2.28 times faster.]_

**Do both libraries give the same numbers?** _[Yes. Both give the same numbers, but the row order is different.]_

## 4.5 — Sorting and taking the top N

In [6]:
pandas_time = measure(
    lambda: df_pandas.sort_values(NUMERIC_COL, ascending=False).head(10)
)
polars_time = measure(
    lambda: df_polars.sort(NUMERIC_COL, descending=True).head(10)
)

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

sort_pandas, sort_polars = pandas_time, polars_time

pandas:     9.08 ms
polars:     2.03 ms
ratio:      4.46x


**Observation:** [Polars was faster than Pandas when sorting the data and selecting the top 10 rows. Polars took 2.03 ms, while Pandas took 9.08 ms, making Polars approximately 4.46 times faster.]

## 4.6 — The Lazy API

This one has no Pandas equivalent, which is the point. Pandas executes every step immediately. Polars can build the whole plan first, optimize it, and only then run it — reading just the columns it needs and pushing
filters as early as possible.

Chain several operations together lazily, then `.collect()`.

In [9]:
def polars_lazy_pipeline():
    return (
        pl.scan_csv(DATA_PATH)
          .filter(pl.col(NUMERIC_COL) > threshold)
          .with_columns((pl.col(NUMERIC_COL) - pl.col(NUMERIC_COL2)).alias("new_col"))
          .group_by(CATEGORY_COL)
          .agg(pl.col("new_col").sum().alias("total"))
          .sort("total", descending=True)
          .collect()
    )

def pandas_equivalent():
    d = pd.read_csv(DATA_PATH)
    d = d[d[NUMERIC_COL] > threshold]
    d = d.assign(new_col=d[NUMERIC_COL] - d[NUMERIC_COL2])
    d = d.groupby(CATEGORY_COL).agg(total=("new_col", "sum"))
    return d.sort_values("total", ascending=False)

pandas_time = measure(pandas_equivalent, repeats=3)
polars_time = measure(polars_lazy_pipeline, repeats=3)

print(f"pandas (step by step): {pandas_time:8.2f} ms")
print(f"polars (lazy + collect): {polars_time:8.2f} ms")
print(f"ratio: {pandas_time/polars_time:8.2f}x")

lazy_pandas, lazy_polars = pandas_time, polars_time

pandas (step by step):    70.90 ms
polars (lazy + collect):     5.90 ms
ratio:    12.01x


**Look at the query plan.** Polars can show you what it decided to do.

In [10]:
plan = (
    pl.scan_csv(DATA_PATH)
      .filter(pl.col(NUMERIC_COL) > threshold)
      .group_by(CATEGORY_COL)
      .agg(pl.col(NUMERIC_COL).sum())
)

print(plan.explain())

AGGREGATE[maintain_order: false]
  [col("Units Sold").sum()] BY [col("Item Type")]
  FROM
  Csv SCAN [Records.csv]
  PROJECT 2/14 COLUMNS
  SELECTION: col("Units Sold").cast(Float64) > 5007.0
  ESTIMATED ROWS: 93868


**Observation:** [The plan shows that Polars reads only 2 out of 14 columns and applies the filter during the CSV scan, before grouping and aggregation.]

## 4.7 — Joining two tables

Build a small lookup table from your data, then join it back.

In [ ]:
# a lookup table: average value per category
lookup_pandas = df_pandas.groupby(CATEGORY_COL, as_index=False)[NUMERIC_COL].mean()
lookup_pandas = lookup_pandas.rename(columns={NUMERIC_COL: "category_average"})

lookup_polars = (
    df_polars.group_by(CATEGORY_COL)
             .agg(pl.col(NUMERIC_COL).mean().alias("category_average"))
)

pandas_time = measure(lambda: df_pandas.merge(lookup_pandas, on=CATEGORY_COL, how="left"))
polars_time = measure(lambda: df_polars.join(lookup_polars, on=CATEGORY_COL, how="left"))

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

join_pandas, join_polars = pandas_time, polars_time

# Row-count check - a join should not change the number of rows here
merged = df_pandas.merge(lookup_pandas, on=CATEGORY_COL, how="left")
print()
print("rows before:", len(df_pandas), "-> after:", len(merged))
print("row count unchanged:", len(merged) == len(df_pandas))

pandas:     8.04 ms
polars:     1.04 ms
ratio:      7.76x

rows before: 100000 -> after: 100000
row count unchanged: True


**Observation:** _[Polars was faster than Pandas when joining the two tables. Polars took 1.04 ms, while Pandas took 8.04 ms, making Polars approximately 7.76 times faster. The number of rows remained unchanged at 100,000 after the join.]_

---
# Part 5 · Results summary

Collect every measurement into one table.

In [5]:
results = pd.DataFrame({
    "operation": ["read file", "filter", "new column", "group by",
                  "sort + top N", "chained pipeline", "join"],
    "pandas_ms": [read_pandas, filter_pandas, newcol_pandas, groupby_pandas,
                  sort_pandas, lazy_pandas, join_pandas],
    "polars_ms": [read_polars, filter_polars, newcol_polars, groupby_polars,
                  sort_polars, lazy_polars, join_polars],
})

results["speedup"] = (results["pandas_ms"] / results["polars_ms"]).round(2)
results["faster"] = np.where(results["speedup"] > 1, "polars", "pandas")

results.round(2)

,operation,pandas_ms,polars_ms,speedup,faster
0,read file,90.90,6.95,13.08,polars
1,filter,2.75,1.77,1.55,polars
2,new column,4.56,1.24,3.68,polars
3,group by,5.95,2.61,2.28,polars
4,sort + top N,9.08,2.03,4.47,polars
5,chained pipeline,70.90,5.90,12.02,polars
6,join,8.04,1.04,7.73,polars


In [6]:
# Save your results so they are in the repository as evidence
import os
os.makedirs("results", exist_ok=True)
results.round(3).to_csv("results/benchmark.csv", index=False)
print("saved to results/benchmark.csv")

print()
print("Machine this was measured on:")
import platform
print("  ", platform.platform())
print("  ", platform.processor() or "processor not reported")

saved to results/benchmark.csv

Machine this was measured on:
   macOS-26.6.2-arm64-arm-64bit-Mach-O
   arm


> Recording the machine matters. Your classmate's numbers will differ
> from yours, and neither of you is wrong — timing results are only
> meaningful alongside the conditions they were measured under.

---
# Part 6 · Your analysis

Write in **your own words**. Short, specific answers are better than long vague ones. Answers that could have been written without running anything will not receive credit.

### Q1 — Where was the difference largest, and where was it smallest?

Name the specific operations and your actual numbers. Do you have a theory about why those particular operations differ most?

_[ The largest speed difference was in reading the file. Pandas took 90.90 ms, while Polars took 6.95 ms, giving a speedup of 13.08x. This may be because Polars can read CSV data using multiple CPU cores.

The smallest difference was in filtering. Pandas took 2.75 ms, while Polars took 1.77 ms, giving a speedup of 1.55x. Both libraries handled this simple operation quickly because the data was already loaded in memory. ]_

### Q2 — Did anything surprise you?

Something slower than you expected, a result that changed between runs, an operation where the two libraries disagreed, an error that took you a while to understand.

_[We were surprised that the speed difference varied so much between operations. Polars was 13.08 times faster at reading the file, but only 1.55 times faster at filtering. This showed us that Polars does not give the same speedup for every task.]_

### Q3 — What was harder to write?

Set the timings aside for a moment. Which library's syntax did you find clearer, and for which operations? Where did you have to look things up?

_[We found Pandas easier to understand, especially for filtering and grouping, because we had used it before. Polars was less familiar, particularly pl.col(), with_columns(), and the lazy API. We needed to check the Polars documentation to understand how these operations work.]_

### Q4 — The marketing claim

Comparisons published online often quote speedups of "5x" or more. Look at the `speedup` column in your Part 5 table.

Did you reproduce those numbers? If not, what might explain the difference? how the published benchmark was run?

_[Yes, but only for some operations. Polars was 13.08x faster at reading, 12.02x faster in the chained pipeline, and 7.73x faster at joining. The other operations had speedups below 5x. Our results show that the speedup depends on the operation, dataset, hardware, and library versions.]_

### Q5 — Your recommendation

You are handed a new project tomorrow. On what basis would you choose one library over the other? Give a concrete situation for each.

_[We would choose based on the dataset size, required operations, and the team's experience. For example, we would use Pandas to analyze a small survey dataset because we know its syntax. We would use Polars to process a large sales CSV with filtering, grouping, and joins, since it was faster in these tasks in our benchmark.]_

### Q6 — What else is out there?

Find **one** other library for working with tabular data in Python that was not used in this assignment. Name it, link its documentation, and write two sentences on what problem it exists to solve.

_[Library: DuckDB

DuckDB helps users analyze tabular data using SQL inside Python without setting up a separate database server. It can query CSV files and Pandas DataFrames directly, making filtering, grouping, and joining data convenient.

Documentation: https://duckdb.org/docs/current/clients/python/overview]_

---
# Submission checklist

- [ ] Part 1: `polars` installed and recorded in `requirements.txt`
- [ ] Part 2: dataset loaded, with 100,000+ rows
- [ ] Part 3: lazy `.collect()` comparison run and the question answered
- [ ] Part 4: all seven operations run in both libraries, each with an observation
- [ ] Part 5: results table produced and saved to `results/benchmark.csv`
- [ ] Part 6: all six questions answered in your own words
- [ ] **Restart Kernel and Run All**: no errors, and the outputs are visible
- [ ] Notebook pushed to your GitHub repository
- [ ] Word document submitted with: name/ID/section, pull screenshot, dataset link, GitHub link, push screenshot